In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *

spark = SparkSession.builder.appName("DataEngineeringProject").getOrCreate()

employee_data = [
(101, "Sravan", "IT", 75000, "Hyderabad"),
(102, "Ravi", "IT", 68000, "Bangalore"),
(103, "Priya", "Analytics", 62000, "Chennai"),
(104, "Kiran", "HR", 90000, "Mumbai"),
(105, "Anjali", "HR", None, "Pune"),      # Missing salary
(106, "Vikram", "Analytics", 98000, None), # Missing city
(101, "Sravan", "IT", 75000, "Hyderabad")  # Duplicate
]

columns = [
    "emp_id",
    "name",
    "department",
    "salary",
    "city"
]

emp_df = spark.createDataFrame(employee_data, columns)

emp_df.show()

+------+------+----------+------+---------+
|emp_id|  name|department|salary|     city|
+------+------+----------+------+---------+
|   101|Sravan|        IT| 75000|Hyderabad|
|   102|  Ravi|        IT| 68000|Bangalore|
|   103| Priya| Analytics| 62000|  Chennai|
|   104| Kiran|        HR| 90000|   Mumbai|
|   105|Anjali|        HR|  NULL|     Pune|
|   106|Vikram| Analytics| 98000|     NULL|
|   101|Sravan|        IT| 75000|Hyderabad|
+------+------+----------+------+---------+



Task 1: Remove Duplicate Records

In [2]:
clean_df = emp_df.dropDuplicates()

clean_df.show()

+------+------+----------+------+---------+
|emp_id|  name|department|salary|     city|
+------+------+----------+------+---------+
|   102|  Ravi|        IT| 68000|Bangalore|
|   103| Priya| Analytics| 62000|  Chennai|
|   101|Sravan|        IT| 75000|Hyderabad|
|   104| Kiran|        HR| 90000|   Mumbai|
|   106|Vikram| Analytics| 98000|     NULL|
|   105|Anjali|        HR|  NULL|     Pune|
+------+------+----------+------+---------+



Task 2: Handle Missing Data

In [3]:
emp_df.show()


+------+------+----------+------+---------+
|emp_id|  name|department|salary|     city|
+------+------+----------+------+---------+
|   101|Sravan|        IT| 75000|Hyderabad|
|   102|  Ravi|        IT| 68000|Bangalore|
|   103| Priya| Analytics| 62000|  Chennai|
|   104| Kiran|        HR| 90000|   Mumbai|
|   105|Anjali|        HR|  NULL|     Pune|
|   106|Vikram| Analytics| 98000|     NULL|
|   101|Sravan|        IT| 75000|Hyderabad|
+------+------+----------+------+---------+



In [4]:
clean_df = clean_df.fillna({
    "salary": 0,
    "city": "Unknown"
})

clean_df.show()

+------+------+----------+------+---------+
|emp_id|  name|department|salary|     city|
+------+------+----------+------+---------+
|   102|  Ravi|        IT| 68000|Bangalore|
|   103| Priya| Analytics| 62000|  Chennai|
|   101|Sravan|        IT| 75000|Hyderabad|
|   104| Kiran|        HR| 90000|   Mumbai|
|   106|Vikram| Analytics| 98000|  Unknown|
|   105|Anjali|        HR|     0|     Pune|
+------+------+----------+------+---------+



Task 3: Join Multiple Datasets

In [5]:
dept_data = [
("IT", "Technology"),
("HR", "Human Resource"),
("Analytics", "Data Analytics")
]

dept_df = spark.createDataFrame(
    dept_data,
    ["department", "department_name"]
)

dept_df.show()

+----------+---------------+
|department|department_name|
+----------+---------------+
|        IT|     Technology|
|        HR| Human Resource|
| Analytics| Data Analytics|
+----------+---------------+



In [6]:
joined_df = clean_df.join(
    dept_df,
    "department",
    "inner"
)

joined_df.show()

+----------+------+------+------+---------+---------------+
|department|emp_id|  name|salary|     city|department_name|
+----------+------+------+------+---------+---------------+
|        IT|   101|Sravan| 75000|Hyderabad|     Technology|
|        IT|   102|  Ravi| 68000|Bangalore|     Technology|
|        HR|   105|Anjali|     0|     Pune| Human Resource|
|        HR|   104| Kiran| 90000|   Mumbai| Human Resource|
| Analytics|   106|Vikram| 98000|  Unknown| Data Analytics|
| Analytics|   103| Priya| 62000|  Chennai| Data Analytics|
+----------+------+------+------+---------+---------------+



Task 4: Optimize Partitions

In [7]:
partition_df = joined_df.repartition(4)

partition_df.show()

+----------+------+------+------+---------+---------------+
|department|emp_id|  name|salary|     city|department_name|
+----------+------+------+------+---------+---------------+
|        HR|   105|Anjali|     0|     Pune| Human Resource|
|        HR|   104| Kiran| 90000|   Mumbai| Human Resource|
|        IT|   101|Sravan| 75000|Hyderabad|     Technology|
| Analytics|   103| Priya| 62000|  Chennai| Data Analytics|
| Analytics|   106|Vikram| 98000|  Unknown| Data Analytics|
|        IT|   102|  Ravi| 68000|Bangalore|     Technology|
+----------+------+------+------+---------+---------------+



Task 5: Create Aggregated Summary Table

In [8]:

summary_df = joined_df.groupBy("department").agg(
    count("*").alias("employee_count"),
    avg("salary").alias("average_salary"),
    max("salary").alias("max_salary"),
    min("salary").alias("min_salary")
)

summary_df.show()

+----------+--------------+--------------+----------+----------+
|department|employee_count|average_salary|max_salary|min_salary|
+----------+--------------+--------------+----------+----------+
|        IT|             2|       71500.0|     75000|     68000|
|        HR|             2|       45000.0|     90000|         0|
| Analytics|             2|       80000.0|     98000|     62000|
+----------+--------------+--------------+----------+----------+



##Complete ETL Pipeline

In [9]:
# Step 1 Remove duplicates
clean_df = emp_df.dropDuplicates()

# Step 2 Handle null values
clean_df = clean_df.fillna({
    "salary": 0,
    "city": "Unknown"
})

# Step 3 Join datasets
joined_df = clean_df.join(
    dept_df,
    "department",
    "inner"
)

# Step 4 Optimize partitions
final_df = joined_df.repartition(4)

# Step 5 Create summary table
summary_df = final_df.groupBy("department").agg(
    count("*").alias("employee_count"),
    avg("salary").alias("average_salary"),
    max("salary").alias("max_salary"),
    min("salary").alias("min_salary")
)

summary_df.show()

+----------+--------------+--------------+----------+----------+
|department|employee_count|average_salary|max_salary|min_salary|
+----------+--------------+--------------+----------+----------+
|        HR|             2|       45000.0|     90000|         0|
| Analytics|             2|       80000.0|     98000|     62000|
|        IT|             2|       71500.0|     75000|     68000|
+----------+--------------+--------------+----------+----------+

